## Execution mode

This public notebook has two explicit execution paths:

- `public` verifies the signed, identifier-free research outputs released under
  `artifacts/` and materializes the corresponding figures and tables without
  CRSP data;
- `full` executes the original empirical notebook and requires the licensed
  inputs documented in `DATA_ACCESS.md`.

The default public path never reads security-level returns, holdings,
identifiers or portfolio weights. No silent fallback is performed.


In [ ]:
# Change to "full" only when the licensed local inputs are available.
import os
from pathlib import Path

RUN_MODE = os.environ.get("MFDRO_RUN_MODE", "public").strip().lower()
if RUN_MODE not in {"public", "full"}:
    raise ValueError("RUN_MODE must be 'public' or 'full'")

_start = Path.cwd().resolve()
PUBLIC_ROOT = next(
    (p for p in (_start, *_start.parents) if (p / "artifacts" / "SHA256SUMS").is_file()),
    None,
)
if PUBLIC_ROOT is None:
    raise FileNotFoundError("run the notebook from within the mt-mfdro repository")

import sys
if str(PUBLIC_ROOT) not in sys.path:
    sys.path.insert(0, str(PUBLIC_ROOT))

print(f"Execution mode: {RUN_MODE.upper()}")
print(f"Repository root: {PUBLIC_ROOT}")


In [ ]:
if RUN_MODE == "public":
    from src.public_results import run_public_notebook

    PUBLIC_REPORT = run_public_notebook("methodology", root=PUBLIC_ROOT)


In [ ]:
if RUN_MODE == "full":
    print("FULL MODE — executing the original licensed-data workflow")


# Methodological illustrations

This notebook gathers the figures used to present the construction of the multi-frequency distributional signal and the conic geometry of the robust allocation problem. The sequence follows the exposition adopted in the methodology and its technical appendix.

In [ ]:
if RUN_MODE == 'full':
    # Set to True to write the publication-ready PDF files.
    EXPORT_OUTPUTS = True

    from pathlib import Path
    import sys

    ROOT = Path.cwd().resolve()
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent
    if not (ROOT / 'Article').exists():
        raise RuntimeError(f'Run from the repository root or notebooks/: {ROOT}')
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))

    print(f'EXPORT_OUTPUTS={EXPORT_OUTPUTS}')


## Common setup

The following cells define the common numerical conventions, graphical charter and deterministic routines used by all illustrations.

In [ ]:
if RUN_MODE == 'full':
    """Deterministic builders for the thesis methodology and appendix figures.

    The figures are displayed in the notebook and written to the output directories
    when ``export_outputs=True``.
    """

    import hashlib
    import json
    import tempfile
    from dataclasses import dataclass
    from pathlib import Path

    import matplotlib.gridspec as gridspec
    import matplotlib.patches as mpatches
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import polars as pl
    import ot
    from matplotlib.lines import Line2D
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
    from scipy.stats import gaussian_kde


    NAVY = "#1f3b5c"
    RUST = "#a23b2e"
    GREEN = "#2e6e4e"
    GREY = "#3c3c3c"


    @dataclass(frozen=True)
    class FigureBuildConfig:
        """Configuration shared by all methodology-figure builders."""

        root: Path
        export_outputs: bool = False
        data_path: Path | None = None

        def resolved_data_path(self) -> Path:
            if self.data_path is not None:
                return self.data_path
            return self.root / "data/processed/nyse_big_caps_pit_daily.parquet"


    class ArticleFigureBuilder:
        """Build, display and optionally export the seven methodological figures."""

        def __init__(self, config: FigureBuildConfig):
            self.config = config
            self.root = Path(config.root).resolve()
            self._tmp = Path(tempfile.mkdtemp(prefix="mthesis_article_figures_"))
            self.records: list[dict[str, object]] = []
            self._aapl: pd.DataFrame | None = None
            self._panel: pd.DataFrame | None = None
            self._set_style()

        @staticmethod
        def _set_style() -> None:
            plt.rcParams.update(
                {
                    "figure.dpi": 120,
                    "savefig.dpi": 300,
                    "font.family": "serif",
                    "font.serif": ["Times New Roman", "DejaVu Serif"],
                    "font.size": 11,
                    "axes.titlesize": 12,
                    "axes.labelsize": 11,
                    "axes.edgecolor": "#333333",
                    "axes.linewidth": 0.8,
                    "axes.spines.top": False,
                    "axes.spines.right": False,
                    "axes.grid": False,
                    "figure.facecolor": "white",
                    "axes.facecolor": "white",
                }
            )

        @staticmethod
        def _sha256(path: Path) -> str:
            digest = hashlib.sha256()
            with path.open("rb") as stream:
                for chunk in iter(lambda: stream.read(1 << 20), b""):
                    digest.update(chunk)
            return digest.hexdigest()

        def _path(self, filename: str, group: str) -> Path:
            if group not in {"Methodology", "Appendix"}:
                raise ValueError(f"unsupported figure group: {group}")
            if self.config.export_outputs:
                output_group = "methodology" if group == "Methodology" else "appendix"
                output_path = self.root / "outputs" / output_group / "images" / filename
            else:
                output_path = self._tmp / filename
            return output_path

        def _finalize(self, fig: plt.Figure, filename: str, group: str) -> Path:
            output_path = self._path(filename, group)
            output_path.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(output_path, bbox_inches="tight", facecolor="white")
            record = {
                "filename": filename,
                "group": group,
                "exported": self.config.export_outputs,
                "output_path": str(output_path),
                "sha256": self._sha256(output_path),
            }
            self.records.append(record)
            plt.show()
            location = output_path.relative_to(self.root) if self.config.export_outputs else output_path.name
            print(f"saved {location}")
            return output_path

        def prepare_market_data(self) -> None:
            """Load only the columns and dates needed by M_01 and M_03."""

            source = self.config.resolved_data_path()
            if not source.exists():
                raise FileNotFoundError(source)

            base = pl.scan_parquet(source).select("PERMNO", "Ticker", "DlyCalDt", "DlyRet")
            aapl = (
                base.filter(
                    (pl.col("Ticker") == "AAPL")
                    & pl.col("DlyCalDt").is_between(
                        pl.datetime(2008, 1, 2), pl.datetime(2024, 12, 31), closed="both"
                    )
                    & pl.col("DlyRet").is_not_null()
                )
                .sort("DlyCalDt")
                .collect()
            )
            if aapl.height < 3_000:
                raise AssertionError(f"unexpectedly short AAPL history: {aapl.height} rows")
            self._aapl = aapl.to_pandas()

            reference = base.filter(
                pl.col("DlyCalDt").is_between(
                    pl.datetime(2017, 1, 1), pl.datetime(2019, 12, 31), closed="both"
                )
                & pl.col("DlyRet").is_not_null()
            )
            counts = (
                reference.group_by("PERMNO")
                .agg(pl.col("DlyCalDt").n_unique().alias("n"))
                .sort(["n", "PERMNO"], descending=[True, False])
                .head(50)
                .collect()
            )
            permnos = counts["PERMNO"].to_list()
            panel = reference.filter(pl.col("PERMNO").is_in(permnos)).collect().to_pandas()
            self._panel = panel
            print(
                f"loaded {source.name}: AAPL={len(self._aapl):,} rows | "
                f"reference panel={len(self._panel):,} rows, {len(permnos)} PERMNOs"
            )

        @staticmethod
        def _resample_single_asset(frame: pd.DataFrame) -> tuple[pd.Series, pd.Series, pd.Series]:
            data = frame.copy()
            data["date"] = pd.to_datetime(data["DlyCalDt"])
            data = data.set_index("date").sort_index()
            daily = data["DlyRet"].dropna()

            def compound(values: pd.Series) -> float:
                return float((1.0 + values).prod() - 1.0)

            weekly = daily.resample("W-FRI").agg([compound, "count"])
            weekly = weekly.loc[weekly["count"] >= 3, "compound"]
            monthly = daily.resample("ME").agg([compound, "count"])
            monthly = monthly.loc[monthly["count"] >= 15, "compound"]
            return daily, weekly, monthly

        @staticmethod
        def _resample_panel(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
            data = frame.copy()
            data["date"] = pd.to_datetime(data["DlyCalDt"])
            daily = data.pivot(index="date", columns="PERMNO", values="DlyRet").sort_index().dropna()

            def compound_frame(values: pd.DataFrame) -> pd.DataFrame:
                return (1.0 + values).prod() - 1.0

            weekly = daily.resample("W-FRI").apply(compound_frame).dropna()
            monthly = daily.resample("ME").apply(compound_frame).dropna()
            common = daily.columns.intersection(weekly.columns).intersection(monthly.columns)
            return daily[common], weekly[common], monthly[common]

        @staticmethod
        def _barycenter(Xd: np.ndarray, Xw: np.ndarray, Xm: np.ndarray) -> np.ndarray:
            return ot.lp.free_support_barycenter(
                measures_locations=[Xd, Xw, Xm],
                measures_weights=[
                    np.ones(len(Xd)) / len(Xd),
                    np.ones(len(Xw)) / len(Xw),
                    np.ones(len(Xm)) / len(Xm),
                ],
                X_init=Xm.copy(),
                b=np.ones(len(Xm)) / len(Xm),
                weights=np.ones(3) / 3,
                numItermax=50,
                stopThr=1e-6,
            )

        def m01_rescaling_aapl(self) -> Path:
            """M_01 — empirical AAPL distributions before and after rescaling."""

            if self._aapl is None:
                self.prepare_market_data()
            daily, weekly, monthly = self._resample_single_asset(self._aapl)
            specs = [
                ("Daily", daily, daily, NAVY, "-", 1.8),
                ("Weekly", weekly, weekly / np.sqrt(5), RUST, "--", 1.8),
                ("Monthly", monthly, monthly / np.sqrt(21), GREEN, ":", 2.0),
            ]

            fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), facecolor="white")
            fig.suptitle(
                "Empirical return distributions by sampling frequency — AAPL",
                fontsize=13,
                y=1.01,
            )
            fig.subplots_adjust(wspace=0.30, bottom=0.16)
            for ax, (title, scaled) in zip(
                axes, [("(a) Before rescaling", False), ("(b) After rescaling", True)]
            ):
                ax.set_title(title, fontsize=11, loc="left", pad=8)
                ax.set_ylabel("Density")
                ax.set_xlabel("Return")
                ax.set_yticks([])
                ax.grid(True, color="#dddddd", linewidth=0.6)
                all_values = np.concatenate(
                    [(scaled_values if scaled else raw).to_numpy() for _, raw, scaled_values, *_ in specs]
                )
                xs = np.linspace(np.percentile(all_values, 0.3), np.percentile(all_values, 99.7), 600)
                for _, raw, scaled_values, color, linestyle, linewidth in specs:
                    values = (scaled_values if scaled else raw).to_numpy()
                    density = gaussian_kde(values, bw_method=0.35)(xs)
                    ax.fill_between(xs, density, alpha=0.05, color=color)
                    ax.plot(xs, density, color=color, lw=linewidth, ls=linestyle)
            handles = [
                Line2D([0], [0], color=color, lw=linewidth, ls=linestyle, label=label)
                for label, _, _, color, linestyle, linewidth in specs
            ]
            fig.legend(
                handles=handles,
                loc="lower center",
                ncol=3,
                fontsize=10,
                frameon=True,
                framealpha=0.9,
                edgecolor="#dddddd",
                bbox_to_anchor=(0.5, -0.02),
            )
            return self._finalize(fig, "M_01_rescaling_aapl.pdf", "Methodology")

        def m02_empirical_measure(self) -> Path:
            """M_02 — schematic conversion of observations into a discrete measure."""

            from matplotlib.patches import FancyArrowPatch

            rng = np.random.default_rng(7)
            fig = plt.figure(figsize=(13, 6), facecolor="white")
            ax_mat = fig.add_axes([0.03, 0.10, 0.32, 0.78])
            ax_arr = fig.add_axes([0.36, 0.36, 0.10, 0.24])
            ax_sc = fig.add_axes([0.49, 0.10, 0.48, 0.78])

            ax_mat.set_xlim(0, 1)
            ax_mat.set_ylim(0, 1)
            ax_mat.axis("off")
            ax_mat.set_title("Rolling window of\nnormalized returns", fontsize=13, pad=8)
            ax_mat.add_patch(
                plt.Rectangle((0.05, 0.02), 0.90, 0.94, linewidth=1.0, edgecolor=GREY, facecolor="white")
            )
            ax_mat.text(0.10, 0.93, r"$\widetilde{X}_{k,\tau} =$", ha="left", va="top", fontsize=13)
            labels = [
                r"$\tilde{r}_{k,1,\tau}^\top$",
                r"$\tilde{r}_{k,2,\tau}^\top$",
                r"$\tilde{r}_{k,3,\tau}^\top$",
                r"$\tilde{r}_{k,4,\tau}^\top$",
            ]
            y_top, row_h, gap = 0.85, 0.12, 0.015
            for index, label in enumerate(labels):
                y0 = y_top - index * (row_h + gap)
                color = "#dbe4ef" if index % 2 == 0 else "#dde9e0"
                ax_mat.add_patch(
                    plt.Rectangle((0.08, y0 - row_h), 0.84, row_h, linewidth=0.6, edgecolor="#999999", facecolor=color)
                )
                ax_mat.text(0.50, y0 - row_h / 2, label, ha="center", va="center", fontsize=11)
            y_dots = y_top - len(labels) * (row_h + gap) - 0.01
            ax_mat.text(0.50, y_dots, r"$\vdots$", ha="center", va="top", fontsize=16)
            y_last = y_dots - 0.09
            ax_mat.add_patch(
                plt.Rectangle((0.08, y_last - row_h), 0.84, row_h, linewidth=0.6, edgecolor="#999999", facecolor="#dbe4ef")
            )
            ax_mat.text(
                0.50,
                y_last - row_h / 2,
                r"$\tilde{r}_{k,T_{k,\tau},\tau}^\top$",
                ha="center",
                va="center",
                fontsize=11,
            )
            ax_mat.text(
                0.50,
                -0.03,
                r"one row = one multivariate return vector in $\mathbb{R}^N$",
                ha="center",
                va="top",
                fontsize=12.5,
                color="#555555",
            )

            ax_arr.set_xlim(0, 1)
            ax_arr.set_ylim(0, 1)
            ax_arr.axis("off")
            ax_arr.add_patch(
                FancyArrowPatch((0.05, 0.55), (0.95, 0.55), arrowstyle="->", mutation_scale=24, linewidth=2.0, color=GREY)
            )
            ax_arr.text(
                0.50,
                0.82,
                "assign uniform\n" + r"mass $\frac{1}{T_{k,\tau}}$" + "\nto each observation",
                ha="center",
                va="bottom",
                fontsize=12,
            )

            px = rng.normal(0.48, 0.14, 30)
            py = rng.normal(0.60, 0.12, 30)
            keep = (px > 0.12) & (px < 0.88) & (py > 0.40) & (py < 0.92)
            px, py = px[keep], py[keep]
            ax_sc.scatter(px, py, s=34, color=NAVY, linewidths=0.5, edgecolors="white", zorder=5)
            highlighted = int(np.argmin((px - 0.668) ** 2 + (py - 0.706) ** 2))
            ax_sc.scatter([px[highlighted]], [py[highlighted]], s=110, color=RUST, linewidths=1.2, edgecolors="white", zorder=7)
            ax_sc.annotate(
                r"$\delta_{\tilde{r}_{k,i,\tau}}$, mass $\dfrac{1}{T_{k,\tau}}$",
                xy=(px[highlighted], py[highlighted]),
                xytext=(px[highlighted] + 0.10, py[highlighted] + 0.16),
                fontsize=13,
                color=RUST,
                arrowprops=dict(arrowstyle="->", color=RUST, lw=1.2, connectionstyle="arc3,rad=-0.2"),
            )
            ax_sc.set_title("Discrete empirical measure", fontsize=13, pad=8)
            ax_sc.text(0.98, 0.97, "each point = one Dirac atom", ha="right", va="top", fontsize=12.5, color="#555555", transform=ax_sc.transAxes)
            ax_sc.text(
                0.50,
                0.11,
                r"$\hat{\mu}_{k,\tau} = \dfrac{1}{T_{k,\tau}}\sum_{i=1}^{T_{k,\tau}}\delta_{\tilde{r}_{k,i,\tau}}$",
                ha="center",
                va="center",
                fontsize=15,
                transform=ax_sc.transAxes,
            )
            ax_sc.set_xticks([])
            ax_sc.set_yticks([])
            ax_sc.set_xlim(0, 1)
            ax_sc.set_ylim(0, 1)
            ax_sc.set_xlabel(r"$r^{(1)}$", labelpad=10)
            ax_sc.set_ylabel(r"$r^{(2)}$", labelpad=10, rotation=0)
            ax_sc.yaxis.set_label_coords(-0.04, 1.0)
            ax_sc.axhline(y=0.25, xmin=0.02, xmax=0.98, color="#cccccc", linewidth=0.6)
            fig.text(0.50, 0.99, "From rolling return observations to an empirical measure", ha="center", va="top", fontsize=14)
            return self._finalize(fig, "M_02_empirical_measure.pdf", "Methodology")

        def m03_barycenter(self) -> Path:
            """M_03 — Wasserstein barycenter as a multi-frequency consensus."""

            if self._panel is None:
                self.prepare_market_data()
            daily, weekly, monthly = self._resample_panel(self._panel)
            Xd = daily.to_numpy() / np.sqrt(1)
            Xw = weekly.to_numpy() / np.sqrt(5)
            Xm = monthly.to_numpy() / np.sqrt(21)
            Xbar = self._barycenter(Xd, Xw, Xm)
            rng = np.random.default_rng(42)
            theta = rng.normal(size=Xd.shape[1])
            theta /= np.linalg.norm(theta)

            fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), facecolor="white")
            fig.subplots_adjust(wspace=0.28, bottom=0.24, top=0.82)
            ax = axes[0]
            layers = [
                ("Daily", Xd, NAVY, "--", 1.6),
                ("Weekly", Xw, RUST, "--", 1.6),
                ("Monthly", Xm, GREEN, "--", 1.6),
                (r"$\mu^*$ barycenter", Xbar, GREY, "-", 2.3),
            ]
            projections = [values @ theta for _, values, *_ in layers]
            lower = min(np.percentile(values, 0.5) for values in projections)
            upper = max(np.percentile(values, 99.5) for values in projections)
            xs = np.linspace(lower, upper, 500)
            for (label, _, color, linestyle, linewidth), values in zip(layers, projections):
                ax.plot(xs, gaussian_kde(values, bw_method=0.4)(xs), color=color, lw=linewidth, ls=linestyle)
            ax.set_xlabel(r"Projected return $\;\theta^\top \tilde{r}$")
            ax.set_ylabel("Density")
            ax.set_title("(a)  Sliced projection & barycenter", fontsize=12, loc="left", pad=6)
            ax.set_yticks([])

            ax = axes[1]
            centers = np.array([[-2.0, 1.4], [2.2, 1.2], [0.1, -2.0]])
            sizes = [150, 90, 36]
            colors = [NAVY, RUST, GREEN]
            spreads = [0.32, 0.38, 0.28]
            clouds = []
            for size, color, center, spread in zip(sizes, colors, centers, spreads):
                points = rng.normal(size=(size, 2)) * spread + center
                clouds.append(points)
                ax.scatter(points[:, 0], points[:, 1], color=color, alpha=0.50, s=16)
            barycenter_location = centers.mean(axis=0)
            barycenter_points = rng.normal(size=(36, 2)) * 0.26 + barycenter_location
            ax.scatter(barycenter_points[:, 0], barycenter_points[:, 1], color=GREY, alpha=0.9, s=28, marker="D", edgecolors="white", linewidths=0.5)
            for points, color in zip(clouds, colors):
                ax.annotate(
                    "",
                    xy=barycenter_points.mean(axis=0),
                    xytext=points.mean(axis=0),
                    arrowprops=dict(arrowstyle="->", color=color, lw=1.2, connectionstyle="arc3,rad=0.10", shrinkA=4, shrinkB=25),
                )
            ax.set_title(r"(b)  Geometric consensus  ($\lambda_k = 1/3$)", fontsize=12, loc="left", pad=6)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlabel(r"$PC_1$")
            ax.set_ylabel(r"$PC_2$")

            fig.suptitle("Wasserstein barycenter as a multi-frequency consensus", fontsize=13, y=0.98)
            handles = [
                mpatches.Patch(color=NAVY, label="Daily"),
                mpatches.Patch(color=RUST, label="Weekly"),
                mpatches.Patch(color=GREEN, label="Monthly"),
                Line2D([0], [0], color=GREY, lw=2.2, label=r"$\mu^*$ barycenter (panel a)"),
                Line2D([0], [0], color=GREY, lw=0, marker="D", markerfacecolor=GREY, markersize=6, label=r"$\mu^*$ barycenter (panel b)"),
            ]
            fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=10, frameon=True, framealpha=0.9, edgecolor="#dddddd", bbox_to_anchor=(0.5, -0.06))
            return self._finalize(fig, "M_03_barycenter.pdf", "Methodology")

        def m04_curse_dimensionality(self) -> Path:
            """M_04 — deterministic curse-of-dimensionality illustration."""

            rng = np.random.RandomState(42)
            n_points = 50
            fig = plt.figure(figsize=(14.5, 4.1), facecolor="white")
            fig.subplots_adjust(wspace=0.48, bottom=0.22, top=0.78, right=0.98, left=0.045)

            ax = fig.add_subplot(1, 4, 1)
            p1 = rng.uniform(0, 1, n_points)
            ax.scatter(p1, np.zeros(n_points), color=NAVY, alpha=0.75, s=20)
            ax.axhline(0, color="#999", lw=0.8)
            ax.set_xlim(-0.05, 1.05)
            ax.set_ylim(-0.3, 0.3)
            ax.set_yticks([])
            ax.set_xticks([0, 0.5, 1])
            ax.set_xlabel(r"$x$")
            ax.set_title("(a)  Dimension 1\n" + r"dense  ($n=50$)", fontsize=11, loc="left", pad=6)

            ax = fig.add_subplot(1, 4, 2)
            p2 = rng.uniform(0, 1, (n_points, 2))
            ax.scatter(p2[:, 0], p2[:, 1], color=NAVY, alpha=0.75, s=20)
            ax.set_xlim(-0.05, 1.05)
            ax.set_ylim(-0.05, 1.05)
            ax.set_xticks([0, 0.5, 1])
            ax.set_yticks([0, 0.5, 1])
            ax.set_xlabel(r"$x_1$")
            ax.set_ylabel(r"$x_2$")
            ax.set_title("(b)  Dimension 2\n" + r"sparser  ($n=50$)", fontsize=11, loc="left", pad=6)

            ax = fig.add_subplot(1, 4, 3, projection="3d")
            p3 = rng.uniform(0, 1, (n_points, 3))
            ax.scatter(p3[:, 0], p3[:, 1], p3[:, 2], color=NAVY, alpha=0.6, s=18)
            ax.set_box_aspect([1, 1, 1])
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.set_zlim(0, 1)
            ax.set_xticks([0, 1])
            ax.set_yticks([0, 1])
            ax.set_zticks([0, 1])
            ax.set_xlabel(r"$x_1$", fontsize=9, labelpad=1)
            ax.set_ylabel(r"$x_2$", fontsize=9, labelpad=1)
            ax.set_zlabel(r"$x_3$", fontsize=9, labelpad=4)
            ax.set_title("(c)  Dimension 3\n" + r"very sparse  ($n=50$)", fontsize=11, pad=6)
            ax.view_init(elev=20, azim=45)

            ax = fig.add_subplot(1, 4, 4)
            dimensions = np.arange(1, 21)
            nearest = []
            for dimension in dimensions:
                points = rng.uniform(0, 1, (n_points, dimension))
                distances = np.sqrt(((points[:, None, :] - points[None, :, :]) ** 2).sum(-1))
                np.fill_diagonal(distances, np.inf)
                nearest.append(distances.min(1).mean())
            ax.plot(dimensions, nearest, color=RUST, lw=2.0, marker="o", ms=3.5)
            for dimension in (1, 2, 3):
                ax.scatter([dimension], [nearest[dimension - 1]], color=NAVY, s=45, edgecolors="white", linewidths=0.8)
            ax.set_xlabel("Dimension $d$")
            ax.set_ylabel("Mean nearest-neighbor distance")
            ax.set_title("(d)  Coverage law\n" + r"($n=50$ fixed)", fontsize=11, loc="left", pad=6)
            ax.set_xticks([1, 5, 10, 15, 20])
            ax.grid(True, color="#dddddd", lw=0.6, axis="y")
            ax.annotate(
                "points drift apart\nas $d$ grows",
                xy=(8, nearest[7]),
                xytext=(11.5, 0.28),
                fontsize=9,
                color=GREY,
                arrowprops=dict(arrowstyle="->", color=GREY, lw=0.9, connectionstyle="arc3,rad=0.25"),
            )
            fig.suptitle("The curse of dimensionality", fontsize=13, y=1.0)
            return self._finalize(fig, "M_04_curse_dimensionality.pdf", "Methodology")

        @staticmethod
        def _projection_sample() -> tuple[np.ndarray, np.ndarray, np.ndarray]:
            rng = np.random.RandomState(7)
            points_a = rng.randn(25, 2) @ np.array([[0.7, 0.3], [0.1, 0.5]]) + [0.2, 0.15]
            points_b = rng.randn(25, 2) @ np.array([[0.5, 0.1], [0.3, 0.7]]) + [-0.2, -0.15]
            theta = np.array([1.0, 1.0])
            theta /= np.linalg.norm(theta)
            return points_a, points_b, theta

        def m05_sliced_projection(self) -> Path:
            """M_05 — projection from R2 to one dimension."""

            points_a, points_b, theta = self._projection_sample()
            fig = plt.figure(figsize=(14, 6), facecolor="white")
            fig.subplots_adjust(wspace=0.38, bottom=0.10, hspace=0.15, top=0.88)
            grid = fig.add_gridspec(2, 2, height_ratios=[5, 1], width_ratios=[1.1, 1.2])
            ax = fig.add_subplot(grid[0, 0])
            ax1d = fig.add_subplot(grid[1, 0])
            ax2 = fig.add_subplot(grid[0, 1])
            fig.add_subplot(grid[1, 1]).set_visible(False)
            limit = 1.8
            origin = np.array([-limit, -limit])
            ax.scatter(points_a[:, 0], points_a[:, 1], color=NAVY, alpha=0.70, s=22)
            ax.scatter(points_b[:, 0], points_b[:, 1], color=RUST, alpha=0.70, s=22)
            for points, color in ((points_a, NAVY), (points_b, RUST)):
                for point in points:
                    foot = origin + np.dot(point - origin, theta) * theta
                    ax.annotate("", xy=foot, xytext=point, arrowprops=dict(arrowstyle="->", color=color, lw=0.6, alpha=0.35, mutation_scale=6))
            ax.annotate("", xy=(limit * 0.98, limit * 0.98), xytext=origin, arrowprops=dict(arrowstyle="->", color=GREY, lw=1.8))
            ax.text(origin[0] + theta[0] * 0.95 - 0.35, origin[1] + theta[1] * 0.95 - 0.15, r"$\theta$", fontsize=14, color=GREY, ha="right", va="top")
            ax.scatter(*origin, color=GREY, s=30)
            ax.set_xlim(-limit, limit)
            ax.set_ylim(-limit, limit)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlabel(r"$x_1$")
            ax.set_ylabel(r"$x_2$")
            ax.set_title(r"(a)  Two distributions in $\mathbb{R}^2$ — projection onto $\theta$", fontsize=12, loc="left", pad=8)

            proj_a = points_a @ theta
            proj_b = points_b @ theta
            lower = min(proj_a.min(), proj_b.min()) - 0.3
            upper = max(proj_a.max(), proj_b.max()) + 0.3
            ax1d.annotate("", xy=(upper, 0), xytext=(lower, 0), arrowprops=dict(arrowstyle="->", color="#444", lw=1.5))
            ax1d.text(lower - 0.08, 0, r"$\theta$", fontsize=12, color="#444", ha="right", va="center")
            ax1d.scatter(proj_a, np.zeros(25), color=NAVY, s=20, alpha=0.8)
            ax1d.scatter(proj_b, np.zeros(25), color=RUST, s=20, alpha=0.8)
            ax1d.set_xlim(lower - 0.4, upper + 0.1)
            ax1d.set_ylim(-0.3, 0.3)
            ax1d.set_yticks([])
            ax1d.set_xticks([])
            for spine in ax1d.spines.values():
                spine.set_visible(False)

            xs = np.linspace(lower - 0.6, upper + 0.6, 400)
            for projection, color, baseline, label in (
                (proj_a, NAVY, 0.55, r"$\theta^\top X_{\mu_1}$"),
                (proj_b, RUST, 0.10, r"$\theta^\top X_{\mu_2}$"),
            ):
                density = gaussian_kde(projection, bw_method=0.4)
                values = density(xs)
                normalized = values / values.max() * 0.28
                center = projection.mean()
                center_height = baseline + float(density(np.array([center]))[0]) / float(values.max()) * 0.28
                ax2.fill_between(xs, baseline, baseline + normalized, alpha=0.07, color=color)
                ax2.plot(xs, baseline + normalized, color=color, lw=2.1)
                ax2.text(center, center_height + 0.02, label, color=color, fontsize=10, ha="center", va="bottom")
            ax2.set_xlabel(r"$\theta^\top x$")
            ax2.set_title(r"(b)  1D projected distributions — $W_2$ by sorting quantiles", fontsize=12, loc="left", pad=8)
            ax2.set_yticks([])
            ax2.set_ylim(0, 0.95)
            ax2.set_xlim(lower - 0.6, upper + 0.6)
            fig.suptitle("Sliced Wasserstein: projection onto a direction reduces the problem to 1D", fontsize=14, y=0.99)
            fig.legend(
                handles=[mpatches.Patch(color=NAVY, label=r"$\mu_1$"), mpatches.Patch(color=RUST, label=r"$\mu_2$")],
                loc="lower center",
                ncol=2,
                fontsize=10,
                frameon=True,
                framealpha=0.9,
                edgecolor="#dddddd",
                bbox_to_anchor=(0.5, 0.08),
            )
            return self._finalize(fig, "M_05_sliced_projection.pdf", "Methodology")

        def m06_w2_quantile(self) -> Path:
            """M_06 — one-dimensional Wasserstein distance by quantile matching."""

            points_a, points_b, theta = self._projection_sample()
            proj_a = points_a @ theta
            proj_b = points_b @ theta
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4), facecolor="white")
            fig.subplots_adjust(wspace=0.30, bottom=0.15, top=0.82)
            xs = np.linspace(min(proj_a.min(), proj_b.min()) - 0.6, max(proj_a.max(), proj_b.max()) + 0.6, 400)
            for projection, color, label in (
                (proj_a, NAVY, r"$\theta^\top X_{\mu_k}$"),
                (proj_b, RUST, r"$\theta^\top X_{\mu^*}$"),
            ):
                density = gaussian_kde(projection, bw_method=0.4)(xs)
                ax1.fill_between(xs, density, alpha=0.08, color=color)
                ax1.plot(xs, density, color=color, lw=2.1, label=label)
            ax1.set_xlabel(r"$\theta^\top x$")
            ax1.set_ylabel("Density")
            ax1.set_title("(a)  Projected densities", fontsize=12, loc="left", pad=8)
            ax1.set_yticks([])
            ax1.set_xlim(xs[0], xs[-1])
            ax1.legend(fontsize=10, frameon=True, framealpha=0.9, edgecolor="#dddddd", loc="upper left")

            u = np.linspace(0, 1, 200)
            quantile_a = np.quantile(proj_a, u)
            quantile_b = np.quantile(proj_b, u)
            w2_squared = np.trapezoid((quantile_a - quantile_b) ** 2, u)
            ax2.fill_between(u, quantile_a, quantile_b, alpha=0.07, color=GREY)
            ax2.plot(u, quantile_a, color=NAVY, lw=2.2, label=r"$F_{\mu_k}^{-1}(u)$")
            ax2.plot(u, quantile_b, color=RUST, lw=2.2, label=r"$F_{\mu^*}^{-1}(u)$")
            for level in np.linspace(0.08, 0.92, 6):
                qa, qb = np.quantile(proj_a, level), np.quantile(proj_b, level)
                ax2.plot([level, level], [qa, qb], color=GREY, lw=0.8, alpha=0.40)
            ax2.text(
                0.97,
                0.06,
                fr"$W_2^2 = {w2_squared:.3f}$",
                transform=ax2.transAxes,
                fontsize=11,
                color=GREY,
                ha="right",
                va="bottom",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="#dddddd"),
            )
            ax2.set_xlabel(r"quantile level $u$")
            ax2.set_ylabel(r"$F^{-1}(u)$")
            ax2.set_title(r"(b)  Quantile functions — $W_2^2=\int_0^1|F_{\mu_k}^{-1}-F_{\mu^*}^{-1}|^2\,du$", fontsize=12, loc="left", pad=8)
            ax2.set_xlim(0, 1)
            ax2.set_xticks([0, 0.25, 0.5, 0.75, 1])
            ax2.legend(fontsize=10, frameon=True, framealpha=0.9, edgecolor="#dddddd", loc="upper left")
            fig.suptitle("One-dimensional optimal transport by quantile matching", fontsize=13, y=1.0)
            return self._finalize(fig, "M_06_w2_quantile.pdf", "Methodology")

        def b01_cones(self) -> Path:
            """B_01 — exact rendering retained from 00_Appendix.ipynb."""

            import numpy as np
            import matplotlib.pyplot as plt
            import matplotlib.gridspec as gridspec
            from matplotlib.lines import Line2D
            from mpl_toolkits.mplot3d import Axes3D
        
            C_CONE = '#2D3FE7'
            C_IN   = '#0A7D44'
            C_OUT  = '#B8001F'
            C_SEG  = '#C45E00'
        
            T_MAX    = 4.0
            R_FACTOR = 1.4
            AZIM     = 35
        
            fig = plt.figure(figsize=(16, 8.5), facecolor='white')
        
            fig.text(0.216, 0.97, r'$\mathcal{Q}^2$ : $|x| \leq t$',
                     ha='center', va='top', fontsize=15, fontweight='bold')
            fig.text(0.733, 0.97, r'$\mathcal{Q}^3$ : $\|(x_1,x_2)\|_2 \leq t$',
                     ha='center', va='top', fontsize=15, fontweight='bold')
        
            gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1.45], wspace=0.02,
                                   left=0.02, right=0.98, top=0.90, bottom=0.14)
        
            # ══════════════════════════════════════════════
            # Panneau gauche : cône 2D
            # ══════════════════════════════════════════════
            ax1 = fig.add_subplot(gs[0])
        
            x_cone = np.linspace(-T_MAX, T_MAX, 600)
            ax1.fill_between(x_cone, np.abs(x_cone), T_MAX, alpha=0.05, color=C_CONE)
            ax1.plot(x_cone, np.abs(x_cone), color=C_CONE, linewidth=2.2)
        
            kw = dict(arrowstyle='->', color='#444', lw=1.0)
            ax1.annotate('', xy=( T_MAX+0.4, 0),  xytext=(-(T_MAX+0.4), 0), arrowprops=kw)
            ax1.annotate('', xy=( 0, T_MAX+0.4),  xytext=(0, -0.2),          arrowprops=kw)
            ax1.text( T_MAX+0.45,  0.09, r'$x$', fontsize=14, color='#222')
            ax1.text( 0.10, T_MAX+0.42, r'$t$', fontsize=14, color='#222')
            ax1.text(-0.28, -0.32, r'$0$', fontsize=12, color='#444')
        
            for tv in [1, 2, 3, 4]:
                ax1.plot([-0.08, 0.08], [tv, tv], color='#444', lw=0.8)
                ax1.text(-0.25, tv, str(tv), fontsize=9, color='#555', va='center', ha='right')
        
            ax1.text(-2.5, 2.8, r'$t = |x|$', fontsize=11, color=C_CONE,
                     ha='center', rotation=45, fontweight='bold')
            ax1.text( 2.5, 2.8, r'$t = |x|$', fontsize=11, color=C_CONE,
                     ha='center', rotation=-45, fontweight='bold')
        
            ax1.scatter([1.5], [2.5], color=C_IN, zorder=6, s=85,
                        edgecolors='white', linewidth=1.0)
            ax1.annotate(r'$(1.5,\,2.5)\ \checkmark$',
                         xy=(1.5, 2.5), xytext=(1.5, 2.72),
                         fontsize=10, color=C_IN, fontweight='semibold',
                         ha='center', va='bottom')
        
            ax1.scatter([2.5], [1.5], color=C_OUT, zorder=6, s=85,
                        edgecolors='white', linewidth=1.0)
            ax1.annotate(r'$(2.5,\,1.5)\ \times$',
                         xy=(2.5, 1.5), xytext=(2.5, 1.72),
                         fontsize=10, color=C_OUT, fontweight='semibold',
                         ha='center', va='bottom')
        
            p1 = np.array([-1.5, 2.0]);  p2 = np.array([0.8, 3.4])
            ax1.scatter(*p1, color=C_SEG, zorder=6, s=85, edgecolors='white', linewidth=1.0)
            ax1.scatter(*p2, color=C_SEG, zorder=6, s=85, edgecolors='white', linewidth=1.0)
            ax1.plot([p1[0], p2[0]], [p1[1], p2[1]], color=C_SEG, linewidth=2.2, linestyle='--')
            ax1.text(-1.1, 2.85, r'segment $\subset \mathcal{Q}^2$',
                     fontsize=9.5, color=C_SEG, ha='center', style='italic', fontweight='bold')
        
            ax1.set_xlim(-(T_MAX+0.8), T_MAX+1.0)
            ax1.set_ylim(-0.55, T_MAX+0.8)
            ax1.axis('off')
        
            # ══════════════════════════════════════════════
            # Panneau droit : cône 3D
            # ══════════════════════════════════════════════
            ax2 = fig.add_subplot(gs[1], projection='3d')
        
            theta  = np.linspace(0, 2*np.pi, 140)
            t_vals = np.linspace(0, T_MAX, 80)
            T_m, Theta = np.meshgrid(t_vals, theta)
            X1 = R_FACTOR * T_m * np.cos(Theta)
            X2 = R_FACTOR * T_m * np.sin(Theta)
        
            ax2.plot_surface(X1, X2, T_m, alpha=0.02, color=C_CONE,
                             linewidth=0, antialiased=True)
        
            for i in range(12):
                ang = 2 * np.pi * i / 12
                tl = np.linspace(0, T_MAX, 60)
                ax2.plot(R_FACTOR*tl*np.cos(ang), R_FACTOR*tl*np.sin(ang), tl,
                         color=C_CONE, linewidth=0.5, alpha=0.22)
        
            # Cercles + labels graduations à droite/extérieur
            # Dans matplotlib 3D avec azim=35 : x2 pointe vers la droite de l'image
            # → labels à droite = grand x2 positif, x1 ≈ 0
            for t_sec, alp in [(1.0, 0.40), (2.0, 0.55), (3.0, 0.70), (4.0, 0.88)]:
                cx = R_FACTOR*t_sec*np.cos(theta)
                cy = R_FACTOR*t_sec*np.sin(theta)
                ax2.plot(cx, cy, np.full_like(theta, t_sec), color=C_CONE, linewidth=1.6, alpha=alp)
                # x2 axis (y dans matplotlib 3D) pointe à droite → label sur bord droit = y positif
                lx = 0.0
                ly = R_FACTOR * t_sec * 1.30   # bord extérieur droit
                ax2.text(lx, ly, t_sec, f'$t={t_sec:.0f}$',
                         fontsize=10, color=C_CONE, fontweight='bold')
        
            # Point vert ∈ Q³
            ax2.scatter([1.4], [1.1], [2.5], color=C_IN, s=80, zorder=8,
                        edgecolors='white', linewidth=0.8, depthshade=False)
            ax2.text(1., 1.1, 2.42, r'$\checkmark$', fontsize=11, color=C_IN, fontweight='bold')
        
            # Point rouge ∉ Q³
            ax2.scatter([4.5], [0.0], [1.0], color=C_OUT, s=80, zorder=8,
                        edgecolors='white', linewidth=0.8, depthshade=False)
            ax2.text(4.7, 0.0, 1.1, r'$\times$', fontsize=11, color=C_OUT, fontweight='bold')
        
            # Segment ⊂ Q³
            s1 = np.array([-0.6,  0.4, 2.5])
            s2 = np.array([ 0.35, -0.35, 2.8])
            lam = np.linspace(0, 1, 30)
            ax2.plot(s1[0]+lam*(s2[0]-s1[0]), s1[1]+lam*(s2[1]-s1[1]), s1[2]+lam*(s2[2]-s1[2]),
                     color=C_SEG, linewidth=2.2, linestyle='--', zorder=7)
            ax2.scatter([s1[0],s2[0]], [s1[1],s2[1]], [s1[2],s2[2]],
                        color=C_SEG, s=70, edgecolors='white', linewidth=0.8, depthshade=False)
            ax2.text(-1.2, 0.4, 2.7, r'seg $\subset\mathcal{Q}^3$',
                     fontsize=9, color=C_SEG, style='italic', fontweight='bold')
        
            ax2.plot([0,0],[0,0],[0, T_MAX+0.8], color='#444', linewidth=1.4)
            ax2.text(0.08, 0.08, T_MAX+1.0, r'$t$', fontsize=14, color='#222')
            ax2.plot([0, R_FACTOR*4.3],[0,0],[0,0], color='#444', linewidth=1.0)
            ax2.plot([0,0],[0, R_FACTOR*4.3],[0,0], color='#444', linewidth=1.0)
            ax2.text(R_FACTOR*4.5, 0, 0, r'$x_1$', fontsize=13, color='#222')
            ax2.text(0, R_FACTOR*4.5, 0, r'$x_2$', fontsize=13, color='#222')
        
            ax2.set_axis_off()
            ax2.view_init(elev=22, azim=AZIM)
            ax2.set_box_aspect([1, 1, 1.5])
        
            fig.canvas.draw()
            pos2 = ax2.get_position()
            ax2.set_position([pos2.x0 - 0.04, pos2.y0 - 0.04,
                              pos2.width + 0.14, pos2.height + 0.10])
        
            legend_handles = [
                Line2D([0],[0], color=C_CONE, lw=2.2,
                       label=r'$\mathcal{Q}^2 = \{(x,t):t\geq|x|\}$'),
                Line2D([0],[0], color=C_IN,  lw=0, marker='o', ms=8,
                       label=r'$(x,t)\in\mathcal{Q}^{2,3}$'),
                Line2D([0],[0], color=C_OUT, lw=0, marker='o', ms=8,
                       label=r'$(x,t)\notin\mathcal{Q}^{2,3}$'),
                Line2D([0],[0], color=C_SEG, lw=2.2, ls='--',
                       label=r'segment $\subset\mathcal{Q}^{2,3}$'),
            ]
            fig.legend(handles=legend_handles,
                       loc='lower center', ncol=4,
                       fontsize=10.5, framealpha=0.92, frameon=True,
                       facecolor='white', edgecolor='black',
                       borderpad=0.7, labelspacing=0.4,
                       bbox_to_anchor=(0.5, 0.01))
        
            return self._finalize(fig, "B_01_cones.pdf", "Appendix")

        def report(self) -> pd.DataFrame:
            """Return the figure inventory and write a manifest on export."""

            report = pd.DataFrame(self.records)
            if report.empty:
                raise AssertionError("no figure has been generated")
            if report["filename"].duplicated().any():
                raise AssertionError("duplicate figure filename")
            expected = {
                "M_01_rescaling_aapl.pdf",
                "M_02_empirical_measure.pdf",
                "M_03_barycenter.pdf",
                "M_04_curse_dimensionality.pdf",
                "M_05_sliced_projection.pdf",
                "M_06_w2_quantile.pdf",
                "B_01_cones.pdf",
            }
            if set(report["filename"]) != expected:
                raise AssertionError("figure-output contract is incomplete")

            if self.config.export_outputs:
                manifest = self.root / "outputs/methodology/methodology_appendix_figure_manifest.json"
                manifest.parent.mkdir(parents=True, exist_ok=True)
                payload = {
                    "schema": "methodology-appendix-figures-v1",
                    "export_outputs": True,
                    "data_path": str(self.config.resolved_data_path()),
                    "figures": self.records,
                }
                manifest.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
                print(f"written {manifest.relative_to(self.root)}")
            else:
                print(f"preview mode: candidates retained temporarily in {self._tmp}")
            return report[
                [
                    "group",
                    "filename",
                    "exported",
                    "output_path",
                    "sha256",
                ]
            ]


In [ ]:
if RUN_MODE == 'full':
    builder = ArticleFigureBuilder(
        FigureBuildConfig(
            root=ROOT,
            export_outputs=EXPORT_OUTPUTS,
        )
    )
    print('self-contained figure builder initialised')


## Data preparation

The empirical illustrations use the point-in-time large-cap panel and load only the dates and variables required for their construction.

In [ ]:
if RUN_MODE == 'full':
    builder.prepare_market_data()


## 1. Multi-frequency rescaling

**Article location:** Methodology — *Mise à l'échelle et construction des mesures empiriques*.

Daily, weekly and monthly returns are placed on a comparable volatility scale before their distributions are compared. Apple (AAPL) provides a univariate illustration of a transformation that remains multivariate in the empirical implementation.

In [ ]:
if RUN_MODE == 'full':
    builder.m01_rescaling_aapl();


## 2. From return matrices to empirical measures

**Article location:** Methodology — *Mise à l'échelle et construction des mesures empiriques*.

Each row of a rolling normalized-return matrix is interpreted as a multivariate observation and becomes a Dirac atom with uniform mass. The diagram summarizes the passage from a data matrix to a discrete empirical distribution on the joint asset space.

In [ ]:
if RUN_MODE == 'full':
    builder.m02_empirical_measure();


## 3. Wasserstein barycenter as a multi-frequency consensus

**Article location:** Methodology — *Barycentre de Wasserstein : consensus multi-fréquence*.

The barycenter aggregates the frequency-specific empirical measures into a common multivariate reference distribution. The displayed projections are illustrative; the barycenter used by the signal is constructed in the original joint asset space.

In [ ]:
if RUN_MODE == 'full':
    builder.m03_barycenter();


## 4. Curse of dimensionality

**Article location:** Methodology — *Dispersion autour du barycentre par Sliced Wasserstein*.

At a fixed sample size, empirical coverage becomes increasingly sparse as dimension grows. This statistical difficulty motivates replacing a direct high-dimensional comparison with a collection of one-dimensional Wasserstein comparisons.

In [ ]:
if RUN_MODE == 'full':
    builder.m04_curse_dimensionality();


## 5. Sliced projection

**Article location:** Methodology — *Réduction dimensionnelle par projections aléatoires*.

A unit direction maps each multivariate observation onto a scalar. Repeating this operation across directions converts the high-dimensional transport comparison into a sequence of tractable one-dimensional problems while retaining multiple views of the original geometry.

In [ ]:
if RUN_MODE == 'full':
    builder.m05_sliced_projection();


## 6. One-dimensional optimal transport

**Article location:** Methodology — *Transport optimal unidimensionnel*.

Once projected, the quadratic Wasserstein distance is obtained by matching equal quantile levels. The shaded separation between the two quantile functions represents the discrepancy that is squared and integrated over the unit interval.

In [ ]:
if RUN_MODE == 'full':
    builder.m06_w2_quantile();


## 7. Geometry of the second-order cone

**Article location:** Appendix B — *Reformulation conique du problème robuste*, subsection *Intuition géométrique du cône du second ordre*.

The two panels illustrate membership and convexity in second-order cones. In three dimensions, every horizontal section at height $t$ is a disk whose radius is governed by the conic norm constraint.

In [ ]:
if RUN_MODE == 'full':
    builder.b01_cones();
